<a href="https://colab.research.google.com/github/George7531/Portfolio_Management/blob/main/Group_19296_MScFE_652_Portfolio_Management_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Code Book:**

##Group Work Project # 2
###GROUP NUMBER: 19296
####Group members:

- Duy Vo
- George Immanuel
- Sing Chun Lee


#### Import Required Packages

In [1]:
# This block import all required python packages for running this notebook
import numpy as np                      # numpy
import matplotlib.pyplot as plt         # for plotting
import math                             # math lib
import pandas as pd                     # for storing data in the DataFrame
import pandas_datareader.data as web
from scipy.optimize import minimize     # for minimizing
import yfinance as yf                   # yfinance library

# Data Preperation and Description

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_datareader.data as web

# Define parameters
tickers = ['TSLA', 'WMT', 'BAC', 'GS', 'LLY', 'MRK', 'GOOG', 'META', 'AAPL', 'XOM']
BENCHMARK = "^GSPC"
in_sample_start = '2023-09-01'
in_sample_end = '2025-09-30'
out_sample_start = '2025-10-01'
out_sample_end = '2025-12-31'

# 1. Download using yfinance
# Using auto_adjust=False
raw_data = yf.download(
    tickers + [BENCHMARK],
    start='2023-08-25', # For initial return
    end='2026-01-01', # the end is exclusive, we should use 2026-01-01 so 2025-12-31 is included
    auto_adjust=False
)

adj_close = raw_data['Close']

# Calculate simple daily percentage returns
daily_returns = adj_close.pct_change().dropna(how='all')

# Drop any trading days where any of the 10 assets have missing data (strict alignment)
daily_returns = daily_returns.dropna()

# In-Sample and Out-of-Sample subsets
returns_in_sample = daily_returns.loc[in_sample_start:in_sample_end]
returns_out_sample = daily_returns.loc[out_sample_start:out_sample_end]

print(f"In-Sample Observations: {len(returns_in_sample)} days")
print(f"Out-of-Sample Observations: {len(returns_out_sample)} days")

# 2. Download Fama-French 5-Factor Daily Data
ff5_daily = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2023-09-01', end='2026-01-01')[0]

# Convert from percentages to decimals
ff5_daily = ff5_daily / 100.0

# Partition factor data
ff_in_sample = ff5_daily.loc[in_sample_start:in_sample_end]
ff_out_sample = ff5_daily.loc[out_sample_start:out_sample_end]

print("\nFama-French Data Sample:")
print(pd.concat([ff_in_sample.head(2), ff_out_sample.tail(3)]))

[*********************100%***********************]  11 of 11 completed


In-Sample Observations: 521 days
Out-of-Sample Observations: 64 days

Fama-French Data Sample:
            Mkt-RF     SMB     HML     RMW     CMA      RF
Date                                                      
2023-09-01  0.0026  0.0085  0.0058  0.0000 -0.0022  0.0002
2023-09-05 -0.0054 -0.0157 -0.0087 -0.0030 -0.0048  0.0002
2025-12-29 -0.0041 -0.0015  0.0007  0.0032  0.0003  0.0002
2025-12-30 -0.0020 -0.0049  0.0029  0.0036  0.0014  0.0002
2025-12-31 -0.0076  0.0006 -0.0010 -0.0007 -0.0004  0.0002


# George's work:


# Step 4c: Detoning

Financial assets are often correlated because they are exposed to some common systematic factors, it could be recession or recent policy changes or whatever that makes all assets in the market to behave like they are all same. A simple example would be the oil syndicate deciding to change the price of fuel, since majority of vehicles use petrol/diesel for locomotion, it will simultaneously affect the prices of all assets from sandals we wear, to colab we use to write this piece down, to colors they use to make floral patterns on their porches(Indian cultural custom) likewise in times of crisis all assets from real estates, to IT stocks, to agricultural companies are affected and heavily correlated and nose dive regardless of their locations and geography in the globe.

How do we deal with this?
When we calculate the correlation matrix of asset returns, the largest eigenvalue and its corresponding eigenvector often capture this common market behaviour. This is called as the market mode. But the things is the market factor that is common among assets would overstate or misreport the actual specific dependency or correlation between companies for example lets say Nvidia and AMD dealing with semiconductors and chips they have some dependency between them that is prior to including the common market factor such as oil price was 0.5 correlation but after including the market factor it might elevate to 0.91 or some other bizzare number that obscures the actual dependency between things we measure. This is where Detoning helps.

Detoning addresses this problem by removing the contribution of the dominant market eigencomponent from the correlation matrix.This allows the remaining correlation structure to reflect more of the relative relationships between assets rather than their common market exposure (López de Prado, 2020).

The steps to detoning are way simpler with statistical packages such as numpy or scipy, I just want to state here the steps in words.

* step1: Take the correlation matrix of our portfolio. Constraint is this correlation matrix must be postive semi-definite meaning its eigen values must be greater than or equal to 0.

* step2: Take eigen decomposition to get eigen vectors and their eigen values$ Correl = V \Lambda V^T $.  Arrange the eigen values and corresponding eigen vectors in descending order to identify the largest contributer(s) as front guardians ($\lambda_1 \geq \lambda_2 \geq \dots \geq \lambda_N$). The largest eigen value ($ \lambda_1$) and its vector($v_1$) typically represent major market factor's contribution.

* step3: In order to remove the effect of this common factor from our correlation matrix, we have to subtract this largest(er) eigen value and vector from  the original covariance matrix such as: $ \hat{correl} = correl - \lambda_1 v_1 v_1^T $, if we want to remove q major factors then we will have to do: $$\hat{Correl} = Correl - \sum_{i=1}^{q} \lambda_i v_i v_i^\top$$

* step4: Then we have to normalize this $ \hat{correl} $ to make it have 1 as its diagonal entries because when we did the transformation(by removing the major contributers by subtraction) the diagonal entries may have gotten altered, so no longer representative of self-correlation, so we do this by taking the diagonal elements of $ \hat{Correl} $ and then take inverse square root of each entry like this:
$$D^{-1/2} = \begin{pmatrix} \frac{1}{\sqrt{\hat{Correl}_{11}}} & 0 & \cdots & 0 \\  0 & \frac{1}{\sqrt{\hat{Correl}_{22}}} & \cdots & 0 \\  \vdots & \vdots & \ddots & \vdots \\  0 & 0 & \cdots & \frac{1}{\sqrt{\hat{Correl}_{NN}}} \end{pmatrix}$$

to finally get our detoned correlated matrix:


$$ Correl_{\text{detoned}} = D^{-1/2} \hat{Correl} D^{-1/2} $$

This detoning technique can be super useful in hierarchical risk clustering or for risk seperation especially during times of crisis when assets in our portfolio would exhibit higher and closer correlations like 0.91,0.93 and so on.. during which time we can remove the common market factor(s) contribution and find how each individual asset are correlated in the absence of that common factor.



# Step 5:

Our plan is use those 10 stocks and check whether reducing noise by denoising, removing dominant market contributions by detoning, and rearranging assets into hierarchical clusters can improve portfolio allocation compared to traditional optimization approach.

we first calculate correlation matrix of our assets and then perform Eigen decompostion on it.

plan:
1. create a function to perform Eigen decomposition on our correlation matrix and to check if it is positive semi definite set a threshold and check if the minimum eigen value is not below it, they should not be below zero.
2. Calculate the correlation matrix.
3. make the correlation matrix symmetric
4. Return the correlation, covariance, and standard deviations

In [3]:
def nearest_correlation_from_psd(A, tol=1e-12):
    A=(np.asarray(A)+np.asarray(A).T)/2
    vals,vecs=np.linalg.eigh(A)
    if vals.min() < -1e-8: raise ValueError('Matrix is materially non-PSD')
    A=(vecs*np.maximum(vals,0))@vecs.T
    diag=np.diag(A)
    if np.any(diag<=tol): raise ValueError('Nonpositive diagonal prevents renormalization')
    corr=A/np.sqrt(np.outer(diag,diag))
    return (corr+corr.T)/2

def sample_structures(returns):
    cov=returns.cov().to_numpy()
    std=np.sqrt(np.diag(cov)); corr=cov/np.outer(std,std)
    return nearest_correlation_from_psd(corr),cov,std

Now denoising:


* we want to check if the eigen value of our correlation matrix is less than threshold defined by $ (1 + \sqrt{q})^2 $ where $ q = \frac{\text{number of columns in our correlation matrix}}{\text{number of rows in our returns dataframe}} $.
* if any of eigen value in the eigen value vector is found being less than or equal to this threshold: $ (1 + \sqrt{q})^2 $ then it is considered noise and we have to replace that eigen value with the average of eigen values in our eigen value vector, and then we reconstruct our correlation with this noise removed eigen values.


In [4]:
def mp_denoise(corr, n_obs):
    n=corr.shape[0]; q=n/n_obs
    if q>=1: raise ValueError('MP implementation assumes number of dates > number of assets')
    threshold=(1+np.sqrt(q))**2
    val,vec=np.linalg.eigh(corr)
    is_noise=val<=threshold
    if not is_noise.any(): raise ValueError('No noise eigenvalues under fixed MP threshold')
    clipped=val.copy(); clipped[is_noise]=val[is_noise].mean()
    den=(vec*clipped)@vec.T
    den=nearest_correlation_from_psd(den)
    return den, {'q':q,'mp_upper':threshold,'n_noise':int(is_noise.sum()),
                 'n_signal':int((~is_noise).sum()),'eig_before':val[::-1],
                 'eig_clipped_pre_normalization':clipped[::-1]}

Detoning:
* I already explained at length about how detoning works, so I'm going to just stroke my brush ever so slightly here.
* Take the correlation matrix, find the eigen decomposition, and remove those high eigen values and their corresponding eigen vectors, and reconstruct our correlation matrix again.

In [5]:
def detone(corr,n_modes=1):
    val,vec=np.linalg.eigh(corr)
    if n_modes<1 or n_modes>=len(val): raise ValueError('Invalid number of market modes')
    indices=np.argsort(val)[::-1][:n_modes]
    market=(vec[:,indices]*val[indices])@vec[:,indices].T
    resid=(corr-market); resid=(resid+resid.T)/2
    if np.min(np.diag(resid))<=1e-12: raise ValueError('Detoning yielded nonpositive residual variance')
    return nearest_correlation_from_psd(resid), {'removed_share':float(val[indices].sum()/val.sum()),
                                                   'removed_eigenvalues':val[indices]}

covariance matrix:
* Multiply our correlation matrix by outer product of volatilities.


In [6]:
def to_cov(corr,std): return (corr*np.outer(std,std))

Put it all together.
first denoise and then detone our correation matrix and then ensure the diagonal of that matrix are all close to 1 indicating self correlation, eigen values are all greater than 0 and detoned correlation matrix should be singular; it's determinant must be zero.

In [8]:
def diagnose_matrix(name,c):
    eig=np.linalg.eigvalsh(c)
    return {'matrix':name,'trace':float(np.trace(c)),'min_eigenvalue':float(eig.min()),
            'max_eigenvalue':float(eig.max()),'condition_number_finite':float(eig.max()/max(eig.min(),1e-14)),
            'mean_off_diagonal_corr':float((np.sum(c)-np.trace(c))/(len(c)*(len(c)-1)))}



In [9]:
train = returns_in_sample.loc['2024-01-01':'2025-09-30', tickers].copy()
train.head(5)

Ticker,TSLA,WMT,BAC,GS,LLY,MRK,GOOG,META,AAPL,XOM
Date,,,,,,,,,,
2024-01-02,-0.000241,0.010403,0.006831,0.006558,0.015920,0.038709,-0.009721,-0.021669,-0.035787,0.023805
2024-01-03,-0.040134,0.000063,-0.010915,-0.016765,0.043060,0.013511,0.005732,-0.005256,-0.007488,0.008402
2024-01-04,-0.002181,-0.009667,0.008053,0.003038,-0.005181,0.019517,-0.016529,0.007693,-0.012700,-0.008719
2024-01-05,-0.001849,-0.006656,0.018639,0.009113,0.006591,0.001795,-0.004709,0.013915,-0.004013,0.003030
2024-01-08,0.012464,0.009827,-0.007842,0.006262,0.012093,0.001365,0.022855,0.019065,0.024175,-0.016662


In [15]:
# Calculate sample covariance and correlation
c_raw, cov_raw, std = sample_structures(train)

# Denoising using Marchenko-Pastur
c_den, mp_info = mp_denoise(c_raw, len(train))

# Detoning using the first principal component
c_tone, tone_info = detone(c_raw)

# Denoising followed by detoning
c_dt, dt_info = detone(c_den)

# Detoning followed by denoising
c_td, td_info = mp_denoise(c_tone, len(train))

# Store matrices
matrices = {
    'Raw': c_raw,
    'Denoised': c_den,
    'Detoned': c_tone,
    'Denoise -> Detoned': c_dt,
    'Detoned -> Denoised': c_td
}

# Generate diagnostic table
diag_table = pd.DataFrame([
    diagnose_matrix(name, c)
    for name, c in matrices.items()
]).set_index('matrix')

print('Marchenko-Pastur details:')
print({k: v for k, v in mp_info.items() if 'eig_' not in k})

print('\nRaw market explained share:',
      round(tone_info['removed_share'], 4))

# Verify the correlation matrices
assert all(
    np.allclose(np.diag(c), 1, atol=1e-8)
    for c in matrices.values()
), "Some matrices do not have unit diagonals"

assert all(
    np.linalg.eigvalsh(c).min() > -1e-8
    for c in matrices.values()
), "Some matrices are not positive semidefinite"

assert np.linalg.eigvalsh(c_tone).min() < 1e-8, \
    "Detoned matrix should be singular"

print("\nAll matrix assertions passed.")

Marchenko-Pastur details:
{'q': 0.0228310502283105, 'mp_upper': np.float64(1.3250299928360603), 'n_noise': 9, 'n_signal': 1}

Raw market explained share: 0.354

All matrix assertions passed.


In [16]:
# Display results
display(diag_table.round(5))

,trace,min_eigenvalue,max_eigenvalue,condition_number_finite,mean_off_diagonal_corr
matrix,,,,,
Raw,10.0,0.20916,3.53958,1.692317e+01,0.25782
Denoised,10.0,0.58933,3.34100,5.669130e+00,0.24764
Detoned,10.0,0.00000,2.16723,2.167234e+14,-0.10518
Denoise -> Detoned,10.0,0.00000,1.14176,1.141755e+14,-0.10854
Detoned -> Denoised,10.0,0.65569,2.03041,3.096580e+00,-0.01963


* In the raw correlation matrix, 35.39% of variance was being captured by the first eigen value (max_eigenvalue of Raw/10 assets), and when we Denoise and then Detone (Denoise -> Detone) we have that reduced to about 11.41%, so the common market factor's effects are effectively reduced, Nice!
* condition_number_finite	Indicates numerical sensitivity; very large values indicate near-singularity, both our detoned and denoised -> detoned correlation matrixes have got 2.16e+14 and 1.14e+14, so their determinants are zero.
* our mean off-diagonal values (covariances) also changed from 0.25782 to -0.10854 in Denoised to Detoned, have stocks have become hedges against one another?